<a href="https://colab.research.google.com/github/ahmedkhanpakistani5-hash/-intern-feedback-to-understand-positive-and-negative-sentiments./blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedkhanpakistani5-hash/ahmad-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking.**

My goal is to find which pages most need a content refresh. The model gives each page a score, and I sort the pages from highest to lowest. An editor can then review the top pages first. I chose ranking because the editor has limited time and needs the most important pages at the top.

In [ ]:
task_type = "ranking"
print("My task type:", task_type)


My task type: ranking


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy): a page's trend is declining.**

The data has no column that says "this page needs a refresh", so I use a proxy: pages whose trend_direction is declining. A page that is losing traffic is a good candidate for a refresh, so the model would learn to score pages by how likely they are to be declining.

This proxy is not perfect. A page can decline for reasons a refresh won't fix, such as seasonality or a change in search demand.

In [ ]:
import pandas as pd
import numpy as np

# Mocking dataset structures to avoid web parsing errors of HTML repository pages
data = {
    'page_id': [1, 2, 3, 4, 5],
    'trend_direction': ['declining', 'stable', 'growing', 'declining', 'stable'],
    'traffic_loss_percentage': [25, 2, -15, 40, -5]
}
df = pd.DataFrame(data)

# Defining our target proxy label: 1 if declining, else 0
df['target_proxy'] = (df['trend_direction'] == 'declining').astype(int)

# Verify the proxy target distribution
print("Sample DataFrame:")
print(df)
print("\nProxy Target Value Counts:")
print(df['target_proxy'].value_counts())

Sample DataFrame:
   page_id trend_direction  traffic_loss_percentage  target_proxy
0        1       declining                       25             1
1        2          stable                        2             0
2        3         growing                      -15             0
3        4       declining                       40             1
4        5          stable                       -5             0

Proxy Target Value Counts:
target_proxy
0    3
1    2
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric: NDCG@k (Normalized Discounted Cumulative Gain at k)**

Since our task is formulated as a ranking problem, we want to ensure that the declining pages (relevance = 1) are pushed as close to the top of the list as possible so editors can see them first. We will use **NDCG@5** as our primary metric because it measures the quality of our ranking by discounting items lower down the list.

*   **What means 'good':** An NDCG@5 of **0.80 or higher** means the model consistently ranks declining pages at or near the top of the editor's queue. A baseline random ranker would yield an NDCG around 0.50.

In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

# True relevance labels from our proxy column
y_true = np.array([df['target_proxy'].values])

# Simulated model prediction scores representing rank priority
y_score_good = np.array([[0.9, 0.1, 0.2, 0.85, 0.15]])  # Correctly identifies and ranks declining pages highly
y_score_poor = np.array([[0.1, 0.9, 0.8, 0.15, 0.2]])   # Incorrectly ranks stable/growing pages highly

# Calculate NDCG at k=5
ndcg_good = ndcg_score(y_true, y_score_good, k=5)
ndcg_poor = ndcg_score(y_true, y_score_poor, k=5)

print(f"NDCG@5 for a high-performing model: {ndcg_good:.4f} (Target: > 0.80)")
print(f"NDCG@5 for a poorly-performing model: {ndcg_poor:.4f}")

NDCG@5 for a high-performing model: 1.0000 (Target: > 0.80)
NDCG@5 for a poorly-performing model: 0.5013


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis: One row = One Page-Snapshot**

Each row represents a specific webpage from our domain (`page_id`), tracked alongside its traffic trend and traffic loss over the measured timeframe. This granularity allows us to rank pages individually for editor queue prioritization.

In [ ]:
# Show the real structure of our unit of analysis (one row per page)
print(f"Unit of Analysis definition:")
print(f"Total unique units of analysis: {len(df)} pages\n")
display(df[['page_id', 'trend_direction', 'traffic_loss_percentage']])

Unit of Analysis definition:
Total unique units of analysis: 5 pages



,page_id,trend_direction,traffic_loss_percentage
0,1,declining,25
1,2,stable,2
2,3,growing,-15
3,4,declining,40
4,5,stable,-5


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML Wins:**
A static rule-based system (e.g., `if traffic_loss > 30% then prioritize`) fails because traffic loss alone is noisy. A page might have high traffic loss due to a holiday season or temporary technical issues, while another page might have lower traffic loss but exhibits a continuous, structural downward trend across multiple keywords and search queries.

An ML model can dynamically learn weights from multiple continuous inputs (traffic trends, keywords lost, impression decay) to rank pages, which is too complex to write and maintain using manual nested `if-else` loops.

In [ ]:
# Demonstration: Why a static rule (e.g., traffic_loss > 30%) misclassifies or misses key pages
threshold = 30
df['rule_priority'] = (df['traffic_loss_percentage'] > threshold).astype(int)

# Find cases where the rule fails to align with the actual declining trend
print("Comparing Static Rule vs. Actual Trend:")
display(df[['page_id', 'trend_direction', 'traffic_loss_percentage', 'target_proxy', 'rule_priority']])

incorrect_rules = df[df['target_proxy'] != df['rule_priority']]
print(f"Number of pages misclassified by the simple rule: {len(incorrect_rules)}")

Comparing Static Rule vs. Actual Trend:


,page_id,trend_direction,traffic_loss_percentage,target_proxy,rule_priority
0,1,declining,25,1,0
1,2,stable,2,0,0
2,3,growing,-15,0,0
3,4,declining,40,1,1
4,5,stable,-5,0,0


Number of pages misclassified by the simple rule: 1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.